# Setting

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os, time  
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
spark = (
   SparkSession.builder
     .appName("PySpark Test")
     .master("local[*]")
    .getOrCreate()
)
 
sc = spark.sparkContext

TASK 1.1 (1 min): Create RDD from ["Spark", "RDD", "HandsOn"] →
print count

In [2]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(nums)
# 2. Basic actions
print("Count:", rdd.count())           # 10
print("First 3:", rdd.take(3))         # [1, 2, 3]
print("Sample:", rdd.takeSample(False, 3))  # Random 3
# MAP: Transform each element
squared = rdd.map(lambda x: x*x)


Count: 10
First 3: [1, 2, 3]
Sample: [1, 9, 8]


In [3]:
nums = ["Spark", "RDD", "HandsOn"]
rdd2 = sc.parallelize(nums)
print(rdd2.collect())

['Spark', 'RDD', 'HandsOn']


Section 2: Transformations (7 mins)

In [4]:
#MAP: Transform each element
squared = rdd.map(lambda x: x*x)
print("Squared:", squared.collect())  # [1, 4, 9, 16...]

# FILTER: Select subset
evens = rdd.filter(lambda x: x%2 == 0)
print("Evens:", evens.collect())      

# Chain them
# [2, 4, 6, 8, 10]
big_squares = rdd.map(lambda x: x*x).filter(lambda x: x > 50)
print("Big squares:", big_squares.collect())

Squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]
Big squares: [64, 81, 100]


In [6]:
cubes = rdd.map(lambda x: x*x*x).filter(lambda x: x > 100)
cubes.collect()

[125, 216, 343, 512, 729, 1000]

In [7]:
total = rdd.reduce(lambda a,b: a+b)
print("Sum:", total)  # 55

Sum: 55


Section 3: Text Processing + flatMap

In [8]:
lines = [
"Apache Spark processes data fast",
"RDD is resilient distributed dataset", 
"Hands-on practice makes perfect",
"Transformations are lazy evaluations"
]
text_rdd = sc.parallelize(lines)
print("Lines:", text_rdd.collect())

Lines: ['Apache Spark processes data fast', 'RDD is resilient distributed dataset', 'Hands-on practice makes perfect', 'Transformations are lazy evaluations']


In [9]:
# FLATMAP: Split + flatten
words = text_rdd.flatMap(lambda line: line.lower().split())
print("All words:", words.collect())
print("First 5:", words.take(5))

All words: ['apache', 'spark', 'processes', 'data', 'fast', 'rdd', 'is', 'resilient', 'distributed', 'dataset', 'hands-on', 'practice', 'makes', 'perfect', 'transformations', 'are', 'lazy', 'evaluations']
First 5: ['apache', 'spark', 'processes', 'data', 'fast']


In [10]:
clean_words = words.filter(lambda w: w != "data").filter(lambda w: len(w) > 3)
print("Xlean words:", clean_words.collect())

Xlean words: ['apache', 'spark', 'processes', 'fast', 'resilient', 'distributed', 'dataset', 'hands-on', 'practice', 'makes', 'perfect', 'transformations', 'lazy', 'evaluations']


In [11]:
long_words = words.filter(lambda w: len(w) > 4)
print("Words with length > 4:", long_words.count())

Words with length > 4: 12


Section 4: Pair RDDs & Word Count (8mins)

In [15]:
# 1. Create (word, 1) pairs
word_pairs = words.map(lambda w: (w, 1))

# 2. Count by word
counts = word_pairs.reduceByKey(lambda a,b: a+b)
print("Word counts:", counts.collect())

# 3. Sort by frequency (descending)
top_words = counts.map(lambda kv: (kv[1], kv[0])) \
                  .sortByKey(ascending=False)
print("Top 3:", top_words.take(3))

Word counts: [('hands-on', 1), ('perfect', 1), ('are', 1), ('spark', 1), ('dataset', 1), ('practice', 1), ('makes', 1), ('lazy', 1), ('processes', 1), ('is', 1), ('transformations', 1), ('evaluations', 1), ('apache', 1), ('data', 1), ('fast', 1), ('rdd', 1), ('resilient', 1), ('distributed', 1)]
Top 3: [(1, 1), (1, 1), (1, 1)]


In [13]:
frequent_words = counts.filter(lambda kv: kv[1] >= 2)
print("Frequent words:", frequent_words.collect())

Frequent words: []


In [20]:
import shutil
import os

# Task 4.2: Save word counts to file
with open("wordcount_output.txt", "w") as f:
    for item in counts.collect():
        f.write(str(item) + "\n")
print("Saved successfully!")



Saved successfully!


Section 5: MINI CHALLENGE

In [22]:
transactions = [
    ("Alice", 100), ("Bob", 200), ("Alice", 50),
    ("Charlie", 70), ("Bob", 30), ("Alice", 150)
]
tx_rdd = sc.parallelize(transactions)

In [24]:
totals = tx_rdd.reduceByKey(lambda a, b: a + b)
totals.collect()

[('Bob', 230), ('Charlie', 70), ('Alice', 300)]

In [34]:
sorted_totals = totals.map(lambda kv: (kv[1], kv[0])).sortByKey(ascending=False).map(lambda kv: (kv[1], kv[0]))

# 3. Print top 2 customers
print("Top 2:", sorted_totals.take(2))

Top 2: [('Alice', 300), ('Bob', 230)]
